# 01 — MT5 Data Ingestion

**Purpose:** Connect to MetaTrader 5, discover the exact DAX symbol available on your broker, download M1 OHLCV bars for DAX and all configured auxiliary symbols, run quality checks, and save to Parquet.

**Prerequisites:**
- MetaTrader 5 desktop app must be running and logged in.
- Python package `MetaTrader5` must be installed: `pip install MetaTrader5`.
- Edit `configs/dax_m1.yaml` with the correct symbol name (or use the discovery cell below).

**Outputs:**
- `data/raw/{SYMBOL}_M1.parquet` — raw M1 bars for DAX
- `data/raw/{SYMBOL}_M1.parquet` — raw M1 bars for each aux symbol
- `data/interim/{SYMBOL}_M{tf}.parquet` — resampled timeframes (M3, M5, M15, M30)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_aux_symbols, get_paths
from src.data.mt5_loader import MT5Loader

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

cfg = load_config()
print("Config loaded. Primary symbol:", get_symbol(cfg))
print("Aux symbols:", get_aux_symbols(cfg))

## 1. Discover DAX symbol

Run this cell first to check which symbol name your broker uses for the DAX. Update `configs/dax_m1.yaml → mt5.symbol` accordingly.

In [ ]:
loader = MT5Loader(cfg)
loader.connect()

# Search for DAX candidates
candidates = loader.find_dax_symbol()
print("\nDAX symbol candidates found:")
for c in candidates:
    print(" -", c)

# Show full symbol list for manual inspection
all_symbols = loader.list_all_symbols()
print(f"\nTotal symbols on broker: {len(all_symbols)}")
print("\nAll symbols (scroll to find DAX):")
display(all_symbols.head(20))

## 2. Download DAX M1 data

Downloads the full historical M1 dataset for the configured symbol. This may take a few minutes depending on the date range.

In [ ]:
symbol = get_symbol(cfg)
print(f"Downloading {symbol} M1...")

df_dax = loader.download_symbol(symbol, "M1")
print(f"\nShape: {df_dax.shape}")
print(f"Date range: {df_dax.index[0]} → {df_dax.index[-1]}")
display(df_dax.head())
display(df_dax.describe())

In [ ]:
path = loader.save_raw(df_dax, symbol, "M1")
print(f"Saved to: {path}")

## 3. Quality checks

In [ ]:
# Check for duplicate timestamps
n_dupes = df_dax.index.duplicated().sum()
print(f"Duplicate timestamps: {n_dupes}")

# Check for time gaps
gaps = MT5Loader.check_gaps(df_dax, expected_freq="1min")
print(f"\nGaps found: {len(gaps)}")
if not gaps.empty:
    display(gaps.head(10))

# Null check
print("\nNull counts:")
print(df_dax.isnull().sum())

# Distribution of spreads
if "spread" in df_dax.columns:
    print(f"\nSpread stats (points):\n{df_dax['spread'].describe()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

df_daily = df_dax["close"].resample("1D").last().dropna()
axes[0].plot(df_daily.index, df_daily.values, lw=0.8)
axes[0].set_title(f"{symbol} Daily Close")
axes[0].set_ylabel("Price")

vol_daily = df_dax["tick_volume"].resample("1D").sum().dropna()
axes[1].bar(vol_daily.index, vol_daily.values, width=0.8, color="steelblue", alpha=0.7)
axes[1].set_title("Daily Volume")
axes[1].set_ylabel("Tick Volume")

plt.tight_layout()
plt.savefig("../reports/dax_overview.png", dpi=120)
plt.show()

## 4. Download auxiliary symbols

In [ ]:
aux_symbols = get_aux_symbols(cfg, enabled_only=True)
print(f"Downloading {len(aux_symbols)} auxiliary symbols: {aux_symbols}")

aux_results = {}
for sym in aux_symbols:
    try:
        df_aux = loader.download_symbol(sym, "M1")
        loader.save_raw(df_aux, sym, "M1")
        aux_results[sym] = df_aux
        print(f"  {sym}: {len(df_aux)} bars — {df_aux.index[0].date()} to {df_aux.index[-1].date()}")
    except Exception as e:
        print(f"  {sym}: FAILED — {e}")

## 5. Resample to derived timeframes

In [ ]:
from pathlib import Path

interim_dir = Path("../data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)

derived_tfs = cfg["mt5"]["timeframes_derived"]  # ["M3", "M5", "M15", "M30"]
print(f"Resampling DAX M1 to: {derived_tfs}")

for tf in derived_tfs:
    df_tf = MT5Loader.resample_ohlcv(df_dax, tf)
    out_path = interim_dir / f"{symbol}_{tf}.parquet"
    df_tf.to_parquet(out_path, engine="pyarrow", compression="snappy")
    print(f"  {symbol} {tf}: {len(df_tf)} bars → {out_path}")

In [ ]:
loader.disconnect()
print("Done. All data saved to data/raw/ and data/interim/")